# ECABSD — Old V3 Training Notebook (33-dim Structural Features)

---

## What This Notebook Does

| Step | Description |
|---|---|
| ✅ Cell 1 | GPU & environment check |
| ✅ Cell 2 | Install all Python dependencies |
| ✅ Cell 3 | Clone ECABSD repository from GitHub |
| ✅ Cell 4 | Mount & verify the pre-processed graph dataset |
| ✅ Cell 5 | Patch config.yaml for Kaggle paths |
| ✅ Cell 6 | Pre-flight checks (graph dims, splits, leakage) |
| 🚀 Cell 7 | **Train Old V3 model — 100 epochs** |
| 📊 Cell 8 | Evaluate on random split (test set) |
| 📊 Cell 9 | Evaluate on homology-filtered split |
| 🔄 Cell 10 | 5-Fold Cross-Validation *(optional — ~4h extra)* |
| 📋 Cell 11 | Final results summary table |
| 💾 Cell 12 | Package outputs for download |
| 🚀 Cell 13 | Push results to GitHub *(optional)* |

---

## Model Architecture (Old V3 — 33-dim)
```
Input: 33-dim structural node features (NO ESM-2)
  ├── 0–19  : amino-acid one-hot
  ├── 20–22 : secondary structure (helix/sheet/coil)
  ├── 23–32 : hydrophobicity, charge, RSA, B-factor, positional encodings...

Encoder    : 6-layer GATv2Conv  (33 → 256-dim, 4 heads, residual connections)
CrossAttn  : 4-head cross-attention (Chain A ← Chain B context)
GlobalPool : mean-pool → GELU projection → fused with local features
Classifier : 3-layer MLP → LayerNorm → ReLU → Dropout(0.3) → Sigmoid
```

**Dataset:** 3,815 pre-processed protein–protein complexes (`.pt` graphs)
**Split:** Train 2,676 / Val 565 / Test 574 — zero leakage verified

**Estimated GPU time:** ~3–5 hours on Kaggle T4

> ⚠️ **Before running:** Enable GPU in *Notebook Settings → Accelerator → GPU T4 x2*

## Cell 1 — GPU & Environment Check

In [ ]:
import subprocess, sys, os

# GPU info
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU detected — enable GPU in Notebook Settings!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Training on CPU will be extremely slow. Enable GPU.')

print(f'\nPython  : {sys.version.split()[0]}')
print(f'Workdir : {os.getcwd()}')

## Cell 2 — Install Dependencies
> Runs once per Kaggle session. Takes ~3–5 minutes.

In [ ]:
import subprocess, sys

def run(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.returncode != 0: print(f'STDERR: {r.stderr[-500:]}')
    return r.returncode

print('=== Installing PyTorch Geometric ===')
run('pip install -q torch-geometric==2.7.0')

print('\n=== Installing project dependencies ===')
run(
    'pip install -q '
    'biopython '
    'pydssp '
    'transformers==4.40.2 '
    'fastapi uvicorn typer '
    'pyyaml scikit-learn tqdm '
    'matplotlib seaborn '
    'python-multipart pandas psutil'
)

print('\n✅ All dependencies installed!')

## Cell 3 — Clone ECABSD Repository
> Set `GITHUB_TOKEN` here if you also want to push results back at the end.

In [ ]:
import os, subprocess, sys

# ── CONFIG — edit these ────────────────────────────────────────────────────────
GITHUB_TOKEN = ''              # optional: paste your GitHub PAT to push results
GITHUB_USER  = 'amanigreeva'
GITHUB_REPO  = 'ECABSD'
WORK_DIR     = '/kaggle/working/ecabsd'
# ──────────────────────────────────────────────────────────────────────────────

if os.path.exists(WORK_DIR):
    print(f'Repo already exists at {WORK_DIR} — pulling latest...')
    subprocess.run(f'git -C {WORK_DIR} pull origin main', shell=True)
else:
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

    print(f'Cloning {GITHUB_REPO}...')
    ret = subprocess.run(f'git clone {clone_url} {WORK_DIR}',
                         shell=True, capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'✅ Cloned to {WORK_DIR}')
    else:
        print(f'ERROR: {ret.stderr}')
        raise RuntimeError('Clone failed — check token or internet connection')

# Set as working directory and Python path
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f'Working directory: {os.getcwd()}')
subprocess.run('ls -la', shell=True)

## Cell 4 — Mount & Verify Pre-Processed Graph Dataset

> **Before running this cell:**
> 1. Upload your `data/processed/` folder + `splits.csv` + `splits_homology.csv` as a Kaggle Dataset
> 2. Name the dataset: `ecabsd-processed-v3`
> 3. Add it to this notebook via *Add Data → Your Datasets → ecabsd-processed-v3*
> 4. It will appear at `/kaggle/input/ecabsd-processed-v3/`
>
> **Dataset contents expected:**
> ```
> ecabsd-processed-v3/
>   ├── processed/          ← 3,815 .pt graph files
>   ├── splits.csv          ← train/val/test split assignments
>   └── splits_homology.csv ← MMseqs2 homology-filtered splits
> ```

In [ ]:
import os, torch

# ── Kaggle dataset mount paths ─────────────────────────────────────────────────
DATASET_ROOT    = '/kaggle/input/ecabsd-processed-v3'
PROCESSED_DIR   = os.path.join(DATASET_ROOT, 'processed')
SPLITS_CSV      = os.path.join(DATASET_ROOT, 'splits.csv')
HOMOLOGY_CSV    = os.path.join(DATASET_ROOT, 'splits_homology.csv')
# ──────────────────────────────────────────────────────────────────────────────

# Verify graph files
if not os.path.exists(PROCESSED_DIR):
    raise FileNotFoundError(
        f'\n❌ Processed graphs not found at {PROCESSED_DIR}\n'
        'Please add the ecabsd-processed-v3 dataset to this notebook.'
    )

pt_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.pt')]
total_mb = sum(os.path.getsize(os.path.join(PROCESSED_DIR, f)) for f in pt_files) / 1024 / 1024

print(f'✅ Graph files found : {len(pt_files)}')
print(f'   Total size        : {total_mb:.1f} MB')
print(f'   Sample files      : {pt_files[:3]}')

# Verify a sample graph
sample = torch.load(os.path.join(PROCESSED_DIR, pt_files[0]), map_location='cpu', weights_only=False)
x_dim  = sample.x.shape[1]
print(f'\n✅ Sample graph: {pt_files[0]}')
print(f'   x shape    : {sample.x.shape}  ← should be (N, 33)')
print(f'   edge_attr  : {sample.edge_attr.shape}  ← should be (E, 5)')
print(f'   y shape    : {sample.y.shape}  ← binding labels')

if x_dim != 33:
    raise ValueError(f'❌ Expected 33-dim node features, got {x_dim}. Wrong dataset version uploaded!')
print(f'\n✅ Node feature dimension confirmed: {x_dim}-dim (correct for Old V3)')

# Verify splits CSV
import pandas as pd
if not os.path.exists(SPLITS_CSV):
    raise FileNotFoundError(f'❌ splits.csv not found at {SPLITS_CSV}')

df = pd.read_csv(SPLITS_CSV)
print(f'\n✅ splits.csv loaded: {len(df)} total entries')
print(df['split'].value_counts().to_string())

# Leakage check
t = set(df[df['split']=='train']['pdb_id'])
v = set(df[df['split']=='val']['pdb_id'])
e = set(df[df['split']=='test']['pdb_id'])
print(f'\nLeakage check: Train∩Val={len(t&v)}, Train∩Test={len(t&e)}, Val∩Test={len(v&e)}')
assert len(t&v)==0 and len(t&e)==0 and len(v&e)==0, '❌ Leakage detected!'
print('✅ Zero leakage confirmed')

# Homology splits
if os.path.exists(HOMOLOGY_CSV):
    dh = pd.read_csv(HOMOLOGY_CSV)
    print(f'\n✅ splits_homology.csv: {len(dh)} entries')
    print(dh['split'].value_counts().to_string())
    ACTIVE_HOMOLOGY_SPLITS = HOMOLOGY_CSV
else:
    print('⚠️  splits_homology.csv not found — homology evaluation will use random splits')
    ACTIVE_HOMOLOGY_SPLITS = SPLITS_CSV

print('\n✅ Dataset verification complete — ready to train!')

## Cell 5 — Patch config.yaml for Kaggle Paths
> Writes `config_v3_old.yaml` — does NOT overwrite your original `config.yaml`.

In [ ]:
import yaml, os

# Load base config from cloned repo
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Override paths for Kaggle ─────────────────────────────────────────────────
cfg['data']['processed_dir']      = PROCESSED_DIR
cfg['data']['splits_csv']         = SPLITS_CSV
cfg['paths']['checkpoints_dir']   = '/kaggle/working/checkpoints'
cfg['paths']['logs_dir']          = '/kaggle/working/logs'
cfg['paths']['results_dir']       = '/kaggle/working/results'

# ── Old V3 model settings — 33-dim structural features ───────────────────────
cfg['model']['esm_dim']           = 33       # CRITICAL — match the .pt files
cfg['model']['hidden_dim']        = 256
cfg['model']['num_gcn_layers']    = 6
cfg['model']['num_heads']         = 4
cfg['model']['dropout']           = 0.3
cfg['model']['edge_feature_dim']  = 5

# ── Training settings ─────────────────────────────────────────────────────────
cfg['training']['epochs']                 = 100
cfg['training']['batch_size']             = 8
cfg['training']['learning_rate']          = 3e-4
cfg['training']['weight_decay']           = 0.005
cfg['training']['warmup_epochs']          = 10
cfg['training']['early_stopping_patience']= 60
cfg['training']['focal_alpha']            = 0.90
cfg['training']['focal_gamma']            = 2.0
cfg['training']['dice_weight']            = 0.40
cfg['training']['loss']                   = 'combined'
cfg['training']['lr_scheduler']           = 'cosine_warmup'
cfg['training']['chain_swap_prob']        = 0.5
cfg['training']['gradient_clip']          = 1.0
cfg['training']['seed']                   = 42
cfg['training']['num_workers']            = 2
cfg['training']['use_wandb']              = False

# ── Web (unused but required by schema) ──────────────────────────────────────
cfg['web']['checkpoint'] = '/kaggle/working/checkpoints/best_model_v3.pt'
# ─────────────────────────────────────────────────────────────────────────────

# Create output directories
for d in ['/kaggle/working/checkpoints', '/kaggle/working/logs', '/kaggle/working/results']:
    os.makedirs(d, exist_ok=True)

# Write Kaggle-specific config
CONFIG_PATH = 'config_v3_old.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f'✅ Written: {CONFIG_PATH}')
print(f'   esm_dim       = {cfg["model"]["esm_dim"]}  ← 33-dim structural (Old V3)')
print(f'   hidden_dim    = {cfg["model"]["hidden_dim"]}')
print(f'   num_gcn_layers= {cfg["model"]["num_gcn_layers"]}')
print(f'   epochs        = {cfg["training"]["epochs"]}')
print(f'   loss          = {cfg["training"]["loss"]}')
print(f'   lr_scheduler  = {cfg["training"]["lr_scheduler"]}')
print(f'   processed_dir = {cfg["data"]["processed_dir"]}')
print(f'   splits_csv    = {cfg["data"]["splits_csv"]}')

## Cell 6 — Pre-Flight Verification
> Loads the dataset through the actual training code path and verifies everything is aligned.

In [ ]:
import torch, yaml
import sys, os
sys.path.insert(0, WORK_DIR)

from data.dataset import BindingSiteDataset, collate_fn
from torch.utils.data import DataLoader

cfg = yaml.safe_load(open(CONFIG_PATH))
mcfg = cfg['model']

print('=== Loading datasets ===')
train_ds = BindingSiteDataset(cfg['data']['processed_dir'], cfg['data']['splits_csv'], split='train')
val_ds   = BindingSiteDataset(cfg['data']['processed_dir'], cfg['data']['splits_csv'], split='val')
test_ds  = BindingSiteDataset(cfg['data']['processed_dir'], cfg['data']['splits_csv'], split='test')

print(f'\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# Verify first sample dimensions
sample = train_ds[0]
x_dim  = sample['data_a'].x.shape[1]
e_dim  = sample['data_a'].edge_attr.shape[1]
print(f'\nFirst sample:')
print(f'  data_a.x shape      : {sample["data_a"].x.shape}  ← should be (N, 33)')
print(f'  data_a.edge_attr    : {sample["data_a"].edge_attr.shape}  ← should be (E, 5)')
print(f'  labels shape        : {sample["labels"].shape}')
print(f'  binding ratio       : {sample["labels"].mean():.3f}')

# Dimension assertions
expected_x = mcfg.get('esm_dim', 33)
assert x_dim == expected_x, f'❌ FATAL: x_dim={x_dim} but config esm_dim={expected_x}'
assert e_dim == 5,          f'❌ FATAL: edge_dim={e_dim} but expected 5'
print(f'\n✅ Dimension check passed: x={x_dim}, edge={e_dim}')

# Model parameter count
from models import ECABSDModel
model = ECABSDModel(
    input_dim     = mcfg.get('esm_dim', 33),
    hidden_dim    = mcfg['hidden_dim'],
    num_heads     = mcfg['num_heads'],
    dropout       = mcfg['dropout'],
    edge_dim      = mcfg.get('edge_feature_dim', 5),
    num_gcn_layers= mcfg.get('num_gcn_layers', 6),
)
total_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ Model instantiated')
print(f'   Total parameters: {total_params:,}')
print(f'   input_dim       : {mcfg.get("esm_dim", 33)}')
print(f'   hidden_dim      : {mcfg["hidden_dim"]}')
print(f'   num_gcn_layers  : {mcfg.get("num_gcn_layers", 6)}')

del model  # free memory before training
print('\n✅ All pre-flight checks passed — ready to train!')

## Cell 7 — 🚀 Train Old V3 Model

| Setting | Value |
|---|---|
| Architecture | 6-layer GATv2 + CrossAttention + GlobalPool |
| Node features | 33-dim structural (no ESM-2) |
| Loss | Focal (α=0.90, γ=2.0) + Soft-Dice (w=0.40) |
| LR schedule | Linear warmup (10 ep) → CosineAnnealing |
| Early stopping | Patience=60 on Val F1 |
| Augmentation | Chain-swap (p=0.5) |
| Max epochs | 100 |

**Best checkpoint saved to:** `/kaggle/working/checkpoints/best_model_v3.pt`  
**Training log saved to:** `/kaggle/working/logs/training_history_v3.json`  
**Estimated time:** ~3–5 hours on GPU T4

In [ ]:
import sys, os
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

from train import run_training

print('=' * 65)
print('  ECABSD Old V3 Training — 33-dim Structural Features')
print('=' * 65)
print(f'  Config     : {CONFIG_PATH}')
print(f'  Checkpoint : /kaggle/working/checkpoints/best_model_v3.pt')
print(f'  Log        : /kaggle/working/logs/training_history_v3.json')
print('=' * 65)

run_training(config_path=CONFIG_PATH)

print('\n✅ Training complete!')

## Cell 8 — 📊 Evaluate on Test Set (Random Split)

In [ ]:
import sys, os, json, yaml
import torch, numpy as np
sys.path.insert(0, WORK_DIR)

from evaluate import run_evaluation

CKPT = '/kaggle/working/checkpoints/best_model_v3.pt'

if not os.path.exists(CKPT):
    print(f'❌ Checkpoint not found at {CKPT}. Run training cell first.')
else:
    print('=== Evaluating on Random Test Split ===')
    metrics_random = run_evaluation(
        config_path     = CONFIG_PATH,
        checkpoint_path = CKPT
    )

    # Save random-split metrics
    out = '/kaggle/working/results/metrics_random_split.json'
    with open(out, 'w') as f:
        json.dump(metrics_random, f, indent=2)
    print(f'\n✅ Random-split metrics saved to: {out}')

## Cell 9 — 📊 Evaluate on Homology-Filtered Split
> Uses `splits_homology.csv` (MMseqs2 ≤30% identity). This is the **publication-standard** metric.

In [ ]:
import sys, os, json, yaml
import torch, numpy as np
sys.path.insert(0, WORK_DIR)

CKPT = '/kaggle/working/checkpoints/best_model_v3.pt'

if not os.path.exists(CKPT):
    print('❌ Checkpoint not found. Run training cell first.')
elif not os.path.exists(ACTIVE_HOMOLOGY_SPLITS):
    print('⚠️  Homology splits CSV not found — skipping homology evaluation.')
else:
    # Temporarily patch config to use homology splits
    cfg_hom = yaml.safe_load(open(CONFIG_PATH))
    cfg_hom['data']['splits_csv'] = ACTIVE_HOMOLOGY_SPLITS

    config_hom_path = 'config_v3_old_homology.yaml'
    with open(config_hom_path, 'w') as f:
        yaml.dump(cfg_hom, f, default_flow_style=False, sort_keys=False)

    print('=== Evaluating on Homology-Filtered Test Split (≤30% identity) ===')
    from evaluate import run_evaluation
    metrics_hom = run_evaluation(
        config_path     = config_hom_path,
        checkpoint_path = CKPT
    )

    # Save homology-split metrics
    out = '/kaggle/working/results/metrics_homology_split.json'
    with open(out, 'w') as f:
        json.dump(metrics_hom, f, indent=2)
    print(f'\n✅ Homology-split metrics saved to: {out}')

## Cell 10 — 🔄 5-Fold Cross-Validation *(Optional)*

> **Estimated time:** ~4–6 hours additional GPU time  
> **Skip this cell** if you only need single-split metrics.  
> Required for peer-reviewed publication (gives `mean ± std`).

Set `RUN_KFOLD = True` to execute.

In [ ]:
import subprocess

RUN_KFOLD = False   # ← set to True to run 5-fold CV

if not RUN_KFOLD:
    print('⏭️  Skipping 5-fold CV (RUN_KFOLD=False)')
    print('   Set RUN_KFOLD = True and re-run this cell to enable.')
else:
    KFOLD_OUT = '/kaggle/working/results/kfold_results.json'

    print('=== 5-Fold Cross-Validation (Homology-Aware Splits) ===')
    print('This will take ~4-6 hours. Training 5 folds × 80 epochs each.')
    print('='*60)

    ret = subprocess.run(
        f'python scripts/train_kfold.py '
        f'--config {CONFIG_PATH} '
        f'--splits {ACTIVE_HOMOLOGY_SPLITS} '
        f'--folds 5 '
        f'--output {KFOLD_OUT} '
        f'--seed 42',
        shell=True,
        cwd=WORK_DIR
    )

    if os.path.exists(KFOLD_OUT):
        import json
        kf = json.load(open(KFOLD_OUT))
        summary = kf.get('summary', {})
        print('\n=== 5-Fold CV Results ===')
        print(f"{'Metric':<15} {'Mean':>8} {'±Std':>8}")
        print('-' * 35)
        for k in ['f1', 'auc_roc', 'auc_pr', 'precision', 'recall', 'mcc']:
            if k in summary:
                print(f"{k:<15} {summary[k]['mean']:>8.4f} {summary[k]['std']:>8.4f}")
        print(f'\n✅ K-fold results saved to: {KFOLD_OUT}')
    else:
        print('⚠️  K-fold output file not found. Check subprocess output above.')

## Cell 11 — 📋 Final Results Summary
> Copy these numbers into `RESULTS.md` before pushing to GitHub.

In [ ]:
import json, os, torch

print('\n' + '=' * 65)
print('  ECABSD Old V3 — FINAL RESULTS SUMMARY')
print('  (33-dim structural features | Re-trained from scratch)')
print('=' * 65)

# Checkpoint metadata
CKPT = '/kaggle/working/checkpoints/best_model_v3.pt'
if os.path.exists(CKPT):
    ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
    print(f'\n📌 Checkpoint Info:')
    print(f'   Best epoch     : {ckpt.get("epoch", "?")+1}')
    print(f'   Best val F1    : {ckpt.get("best_val_f1", 0):.4f}')
    print(f'   Best threshold : {ckpt.get("best_threshold", 0.5):.4f}')

# Random split metrics
rand_path = '/kaggle/working/results/metrics_random_split.json'
if os.path.exists(rand_path):
    m = json.load(open(rand_path))
    print(f'\n📊 Random Split (70/15/15):')
    print(f'   F1-Score  : {m.get("f1", 0):.4f}')
    print(f'   ROC-AUC   : {m.get("auc_roc", 0):.4f}')
    print(f'   PR-AUC    : {m.get("auc_pr", 0):.4f}')
    print(f'   Precision : {m.get("precision", 0):.4f}')
    print(f'   Recall    : {m.get("recall", 0):.4f}')
    print(f'   Accuracy  : {m.get("accuracy", 0):.4f}')
    print(f'   MCC       : {m.get("mcc", 0):.4f}')

# Homology-filtered metrics
hom_path = '/kaggle/working/results/metrics_homology_split.json'
if os.path.exists(hom_path):
    m = json.load(open(hom_path))
    print(f'\n📊 Homology-Filtered (MMseqs2 ≤30% identity) — USE THIS FOR PAPER:')
    print(f'   F1-Score  : {m.get("f1", 0):.4f}')
    print(f'   ROC-AUC   : {m.get("auc_roc", 0):.4f}')
    print(f'   PR-AUC    : {m.get("auc_pr", 0):.4f}')
    print(f'   Precision : {m.get("precision", 0):.4f}')
    print(f'   Recall    : {m.get("recall", 0):.4f}')
    print(f'   Accuracy  : {m.get("accuracy", 0):.4f}')
    print(f'   MCC       : {m.get("mcc", 0):.4f}')

# K-fold results
kfold_path = '/kaggle/working/results/kfold_results.json'
if os.path.exists(kfold_path):
    kf = json.load(open(kfold_path))
    summary = kf.get('summary', {})
    print(f'\n📊 5-Fold Cross-Validation (Homology-Aware):')
    print(f"   {'Metric':<12} {'Mean':>8} {'±Std':>8}")
    print(f"   {'-'*30}")
    for k in ['f1', 'auc_roc', 'auc_pr', 'mcc']:
        if k in summary:
            print(f"   {k:<12} {summary[k]['mean']:>8.4f} {summary[k]['std']:>8.4f}")

print('\n' + '=' * 65)
print('  Copy the numbers above into RESULTS.md before pushing!')
print('=' * 65)

## Cell 12 — 💾 Package Outputs for Download
> All key files will be collected to `/kaggle/working/output/` for download via Kaggle UI.

In [ ]:
import shutil, os

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

files_to_save = [
    ('/kaggle/working/checkpoints/best_model_v3.pt',            'best_model_v3_retrained.pt'),
    ('/kaggle/working/results/metrics_random_split.json',       'metrics_random_split.json'),
    ('/kaggle/working/results/metrics_homology_split.json',     'metrics_homology_split.json'),
    ('/kaggle/working/results/kfold_results.json',              'kfold_results.json'),
    ('/kaggle/working/logs/training_history_v3.json',           'training_history_v3.json'),
    ('config_v3_old.yaml',                                       'config_v3_old.yaml'),
]

print(f'Saving outputs to {OUTPUT_DIR}:')
for src, dst_name in files_to_save:
    dst = os.path.join(OUTPUT_DIR, dst_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  ✅ {dst_name:50s} ({size_mb:.1f} MB)')
    else:
        print(f'  ⚠️  {src} — not found (skipped)')

print(f'\n✅ Done. Download from Kaggle UI: Output tab → /kaggle/working/output/')
print('\nAfter downloading:')
print('  1. Copy best_model_v3_retrained.pt → checkpoints/best_model_v3.pt')
print('  2. Copy training_history_v3.json   → logs/')
print('  3. Update RESULTS.md with metrics from Cell 11')
print('  4. git add -A && git commit -m "results: re-trained Old V3, verified metrics"')

## Cell 13 — 🚀 Push Results to GitHub *(Optional)*
> Only runs if `GITHUB_TOKEN` was set in Cell 3.

In [ ]:
import json, os, subprocess, shutil

if not GITHUB_TOKEN:
    print('⏭️  Skipping push — GITHUB_TOKEN not set in Cell 3.')
    print('   Download results from /kaggle/working/output/ and push manually.')
else:
    os.chdir(WORK_DIR)

    # Configure git identity
    subprocess.run(f'git config user.email "{GITHUB_USER}@users.noreply.github.com"', shell=True)
    subprocess.run(f'git config user.name "{GITHUB_USER}"', shell=True)

    os.makedirs('results', exist_ok=True)
    os.makedirs('logs',    exist_ok=True)

    # Copy results into repo
    copy_pairs = [
        ('/kaggle/working/results/metrics_random_split.json',   'results/metrics_random_split.json'),
        ('/kaggle/working/results/metrics_homology_split.json', 'results/metrics_homology_split.json'),
        ('/kaggle/working/results/kfold_results.json',          'results/kfold_results.json'),
        ('/kaggle/working/logs/training_history_v3.json',       'logs/training_history_v3.json'),
        ('config_v3_old.yaml',                                  'config_v3_old.yaml'),
    ]
    for src, dst in copy_pairs:
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'  Copied: {src} → {dst}')

    # Auto-update RESULTS.md version history entry
    rand_path = 'results/metrics_random_split.json'
    hom_path  = 'results/metrics_homology_split.json'
    ckpt_path_local = '/kaggle/working/checkpoints/best_model_v3.pt'

    import torch
    ckpt  = torch.load(ckpt_path_local, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', 0) + 1

    rand_f1 = json.load(open(rand_path)).get('f1', 0) if os.path.exists(rand_path) else 0
    hom_f1  = json.load(open(hom_path)).get('f1', 0)  if os.path.exists(hom_path)  else 0

    new_entry = (
        f'| **V3-retrained** | **July 2026** | {rand_f1:.4f} (random) / '
        f'{hom_f1:.4f} (homology) | Re-trained from scratch, epoch {epoch}, '
        f'33-dim structural features |\n'
    )

    with open('RESULTS.md', 'a') as f:
        f.write(f'\n<!-- auto-appended by ECABSD_V3_Old_Training notebook -->\n')
        f.write(new_entry)
    print(f'\n  Appended to RESULTS.md: {new_entry.strip()}')

    # Note: checkpoint is large (~50 MB) — git LFS required for pushing .pt files
    subprocess.run('git add results/ logs/ RESULTS.md config_v3_old.yaml', shell=True)
    subprocess.run(
        'git commit -m "results: re-trained Old V3 (33-dim), verified metrics, updated RESULTS.md"',
        shell=True
    )
    subprocess.run('git push origin main', shell=True)
    print('\n✅ Pushed to GitHub!')

## ✅ Training Complete!

### After downloading from Kaggle, do this locally:

```bash
# 1. Replace checkpoint
cp best_model_v3_retrained.pt  ecabsd/checkpoints/best_model_v3.pt

# 2. Replace training log
cp training_history_v3.json    ecabsd/logs/training_history_v3.json

# 3. Copy metrics
cp metrics_random_split.json   ecabsd/results/
cp metrics_homology_split.json ecabsd/results/

# 4. Update RESULTS.md with the numbers from Cell 11

# 5. Commit everything
git add -A
git commit -m "results: Old V3 re-trained from scratch — metrics verified"
git push
```

### Checklist
| Item | Status |
|---|---|
| Old V3 checkpoint re-trained | ✅ |
| Random split metrics verified | ✅ |
| Homology-filtered metrics verified | ✅ |
| RESULTS.md updated | ✅ |
| 5-Fold CV (optional) | ⬜ |

---
**Next step:** Run the Updated V3 notebook (`ECABSD_V3_Updated_Training.ipynb`) for ESM-2 1280-dim training.